# Ancestry-informative marker (AIM) data access — `Ag3`

Ancestry-informative markers (AIMs) are SNP sites specifically chosen because they carry alleles that are highly discriminating between taxa within the *Anopheles gambiae* complex. They underpin *Ag3*'s molecular species calls. This notebook covers the methods for accessing AIM identifiers, AIM site definitions, AIM genotype calls, a heatmap visualisation of AIM genotypes, and the derived AIM/species metadata.

In [1]:
import malariagen_data
ag3 = malariagen_data.Ag3(
    "simplecache::gs://vo_agam_release_master_us_central1",
    simplecache=dict(cache_storage="../../gcs_cache"),
    results_cache="../../results_cache",
)
ag3

/opt/homebrew/Caskroom/miniconda/base/envs/malariagen2/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


<MalariaGEN Ag3 API client>
Storage URL                           : simplecache::gs://vo_agam_release_master_us_central1
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
Results cache                         : /Users/katie.barr/malariagen-data-python/results_cache
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.8.0.post13+b769b728
Client location                       : England, United Kingdom
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
---
Please note that data are subject to terms of use,
for more information see the Vector Observatory website https://www.malariagen.net/vobs/
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.8.0.post13+b769b728/Ag3.html

## `aim_ids`

A read-only property (no parameters) returning the identifiers of the AIM panels available for this data resource. These are the valid values for the `aims` parameter accepted by `aim_variants`, `aim_calls` and `plot_aim_heatmap`. For *Ag3* there are two panels: `gambcolu_vs_arab` (markers discriminating the gambiae/coluzzii group from arabiensis) and `gamb_vs_colu` (markers discriminating gambiae from coluzzii specifically).

**Diagram opportunity:** a flowchart of the AIM species-calling logic — first apply the `gambcolu_vs_arab` panel to split arabiensis from the gambiae/coluzzii group, then apply `gamb_vs_colu` within the latter group to separate gambiae from coluzzii (and flag intermediate/hybrid genotypes) — showing how the two panels combine to produce the `aim_species` calls found in `aim_metadata`. Would fit well here, before the per-panel examples below.

In [2]:
ag3.aim_ids

('gambcolu_vs_arab', 'gamb_vs_colu')

## `aim_variants`

Access the AIM site definitions for one panel: the genome positions of the marker SNPs and, for each, the pair of discriminating alleles (one per taxon being distinguished). Returned as an `xarray.Dataset` with dimensions `variants` (number of AIM sites) and `alleles` (always 2). The only parameter:

- **aims**: which AIM panel to use — one of the values from `aim_ids`, e.g. `"gambcolu_vs_arab"` or `"gamb_vs_colu"`.

The example loads the site definitions for the `gambcolu_vs_arab` panel.

In [3]:
ds_aim_variants = ag3.aim_variants(aims="gambcolu_vs_arab")
ds_aim_variants

<xarray.Dataset> Size: 18kB
Dimensions:           (variants: 2612, alleles: 2)
Coordinates:
    variant_contig    (variants) uint8 3kB dask.array<chunksize=(2612,), meta=np.ndarray>
    variant_position  (variants) int32 10kB dask.array<chunksize=(2612,), meta=np.ndarray>
Dimensions without coordinates: variants, alleles
Data variables:
    variant_allele    (variants, alleles) |S1 5kB dask.array<chunksize=(2612, 2), meta=np.ndarray>
Attributes:
    aims:      gambcolu_vs_arab
    analysis:  20220528
    contigs:   ['2R', '2L', '3R', '3L', 'X']

## `aim_calls`

Access AIM genotype calls: for each sample and each AIM site in a panel, which of the two discriminating alleles (or both, if heterozygous) was observed. Returned as an `xarray.Dataset` with dimensions `variants`, `samples`, `ploidy` (2) and `alleles` (2). Parameters:

- **aims**: which AIM panel to use (see `aim_ids`).
- **sample_sets**: which sample set(s)/release(s) to include.
- **sample_query**: pandas query string to select samples from the metadata.
- **sample_query_options**: extra kwargs passed through to pandas `query()`/`eval()`.

The example loads `gamb_vs_colu` AIM calls for two Burkina Faso sample sets, restricted (via `sample_query`) to samples not already called as arabiensis — since this panel is only informative for distinguishing gambiae from coluzzii.

In [4]:
ds_aim_calls = ag3.aim_calls(
    aims="gamb_vs_colu",
    sample_sets=["AG1000G-BF-A", "AG1000G-BF-B"],
    sample_query="aim_species != 'arabiensis'",
)
ds_aim_calls

Load sample metadata: ⠋ (0:00:00.00)

I0918 18:54:33.525388 13866116 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0918 18:54:33.530318 13866255 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)


<xarray.Dataset> Size: 427kB
Dimensions:           (variants: 700, samples: 280, ploidy: 2, alleles: 2)
Coordinates:
    variant_contig    (variants) uint8 700B dask.array<chunksize=(700,), meta=np.ndarray>
    variant_position  (variants) int64 6kB dask.array<chunksize=(700,), meta=np.ndarray>
    sample_id         (samples) <U24 27kB dask.array<chunksize=(141,), meta=np.ndarray>
Dimensions without coordinates: variants, samples, ploidy, alleles
Data variables:
    call_genotype     (variants, samples, ploidy) int8 392kB dask.array<chunksize=(700, 141, 2), meta=np.ndarray>
    variant_allele    (variants, alleles) |S1 1kB dask.array<chunksize=(700, 2), meta=np.ndarray>
Attributes:
    aims:      gamb_vs_colu
    analysis:  20220528
    contigs:   ['2R', '2L', '3R', '3L', 'X']

## `plot_aim_heatmap`

Plot a heatmap of AIM genotypes — one row per sample, one column per AIM site (faceted into subplots by contig), coloured by genotype (homozygous for taxon 1, heterozygous, homozygous for taxon 2, or missing). This is a quick visual way to see how cleanly samples split between taxa. Shares `aims`, `sample_sets`, `sample_query`, `sample_query_options` with `aim_calls` (used internally to fetch the data), plus:

- **sort**: if `True` (default), order samples by their overall fraction of taxon-2 AIM alleles, so the heatmap gradates from one taxon to the other rather than showing samples in arbitrary order.
- **row_height**: height per sample row, in pixels.
- **xgap**: gap (in pixels) drawn between columns (variants), useful for visually separating individual markers.
- **ygap**: gap (in pixels) drawn between rows (samples).
- **palette**: an optional explicit list of 4 colours (missing, homozygous taxon 1, heterozygous, homozygous taxon 2) overriding the resource's default AIM palette.
- **show**: if `True` (default), display the figure immediately.
- **renderer**: optional Plotly renderer name to use when showing the figure (e.g. `"notebook"`, `"png"`).

The example plots the `gambcolu_vs_arab` heatmap for a sample set known to contain a mix of arabiensis and gambiae/coluzzii samples, with `xgap` set to a non-default value to visually separate the individual marker columns.

In [5]:
ag3.plot_aim_heatmap(
    aims="gambcolu_vs_arab",
    sample_sets="AG1000G-UG",
    xgap=0.5,
)

## `aim_metadata`

Access the derived per-sample AIM/species-calling results as a pandas DataFrame (one row per sample) — the summarised output of applying the AIM panels, including the called `aim_species` (and related fraction/count columns), rather than the raw per-site genotypes returned by `aim_calls`. The only parameter:

- **sample_sets**: which sample set(s)/release(s) to include.

The example loads AIM metadata for the same Uganda sample set used above and tabulates the resulting species calls.

In [6]:
df_aim_meta = ag3.aim_metadata(sample_sets="AG1000G-UG")
df_aim_meta["aim_species"].value_counts()

aim_species
gambiae                             207
arabiensis                           82
intermediate_gambcolu_arabiensis      1
Name: count, dtype: int64